# Metric Analysis — Baseline vs clDice Fine-Tune

Re-runs the width-stratified IoU and connectivity analysis from
notebook `08_metric_analysis.ipynb` **twice** — once for the baseline
model and once for the clDice fine-tuned model — and reports the
delta per bucket.

**Thesis of the experiment:**
- Notebook 08 exposed that aggregate IoU (0.7016) hides failure on
  the narrow bucket (roads ≤ 10 px wide) and that a non-trivial
  fraction of GT road graphs get fragmented.
- clDice loss targets exactly this: it reweights every centerline
  pixel equally, so a 3-px gap in a thin road costs as much as a
  3-px gap in a highway.
- **Success criterion:** we want narrow-bucket IoU and connectivity
  fraction to go up. Aggregate IoU may drop slightly — that's the
  trade-off we accept for topology.

**Caching:** baseline predictions live in `.../metric_analysis/` (from
notebook 08); clDice predictions live in a new `.../metric_analysis_cldice/`.
Re-runs skip already-cached samples.


---
## 1 · Environment (same branch as notebook 08)

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
BRANCH = "metric-analysis"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kagglehub", "scikit-image", "pyarrow"], check=True)
    print("Environment ready.")
else:
    REPO_DIR = None


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

branch = subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git branch:   {branch}")
assert branch == BRANCH, f"Expected branch {BRANCH}, got {branch!r}"


In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


---
## 2 · Two caches, two checkpoints

- `BASELINE_CACHE_DIR` — from notebook 08 (reuse, don't re-infer)
- `CLDICE_CACHE_DIR`   — new; will be filled by this notebook


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/road_segmentation")
else:
    ROOT = Path("data/interim")

BASELINE_CACHE_DIR = ROOT / "metric_analysis"           # already populated by nb 08
CLDICE_CACHE_DIR   = ROOT / "metric_analysis_cldice"    # fresh
for d in (BASELINE_CACHE_DIR, CLDICE_CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"Baseline cache: {BASELINE_CACHE_DIR}")
print(f"clDice cache:   {CLDICE_CACHE_DIR}")


In [ ]:
# ====== POINT AT BOTH CHECKPOINTS ======
BASELINE_CHECKPOINT = ""  # best.pth of the 0.7016 baseline
CLDICE_CHECKPOINT   = ""  # best.pth produced by notebooks/09_train_cldice.ipynb

if IN_COLAB:
    ckpts = {p.parent.name: p for p in Path(ROOT / "checkpoints").rglob("best.pth")}
    if not BASELINE_CHECKPOINT:
        # Prefer the boundary-fine-tuned or the base B4-1024 run
        for key in ("unetplusplus_b4_1024_boundary", "unetplusplus_efficientnet_b4_1024",
                    "unetplusplus_efficientnet_b4"):
            if key in ckpts:
                BASELINE_CHECKPOINT = str(ckpts[key]); break
    if not CLDICE_CHECKPOINT and "unetplusplus_b4_1024_cldice" in ckpts:
        CLDICE_CHECKPOINT = str(ckpts["unetplusplus_b4_1024_cldice"])

print(f"Baseline ckpt: {BASELINE_CHECKPOINT}")
print(f"clDice ckpt:   {CLDICE_CHECKPOINT}")
assert Path(BASELINE_CHECKPOINT).exists(), "Set BASELINE_CHECKPOINT"
assert Path(CLDICE_CHECKPOINT).exists(), "Set CLDICE_CHECKPOINT (run notebook 09 first)"


---
## 3 · Val split (same seed → same samples as notebook 08)

In [ ]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs

train_dir = DEEPGLOBE_DATASET_DIR / "train"
if not (train_dir.exists() and any(train_dir.glob("*_sat.*"))):
    from road_segmentation.data.download import download_dataset
    download_dataset()

pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(pairs, val_ratio=0.15, seed=42)
print(f"Val samples: {len(val_pairs)}")


---
## 4 · Cache predictions for both models

Each block skips samples that are already on disk. If notebook 08
has already filled the baseline cache, the first block is a no-op.


In [ ]:
from road_segmentation.api.inference import InferenceEngine
from road_segmentation.analysis.inference_cache import build_predictions_cache, is_cached

def cache_preds(checkpoint_path, cache_dir, label):
    cached = sum(1 for p in val_pairs if is_cached(p.sample_id, cache_dir))
    print(f"[{label}] cached: {cached}/{len(val_pairs)}")
    if cached >= len(val_pairs):
        return
    engine = InferenceEngine.from_checkpoint(checkpoint_path, device=device)
    stats = build_predictions_cache(
        val_pairs=val_pairs, model=engine.model,
        base_dir=cache_dir, device=device, image_size=engine.image_size,
    )
    print(f"[{label}] done: +{stats.newly_cached} new, {stats.failed} failed, "
          f"{stats.elapsed_s / 60:.1f} min")
    del engine
    if device.type == "cuda": torch.cuda.empty_cache()

cache_preds(BASELINE_CHECKPOINT, BASELINE_CACHE_DIR, "baseline")
cache_preds(CLDICE_CHECKPOINT,   CLDICE_CACHE_DIR,   "clDice")


---
## 5 · Width distribution + bucket edges

Computed once on GT — identical for both models. Re-uses the cached
widths from notebook 08 if present.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")

from road_segmentation.analysis.inference_cache import load_cached
from road_segmentation.analysis.width_metrics import (
    collect_pixel_widths, percentile_edges, FIXED_EDGES,
)

widths_cache = BASELINE_CACHE_DIR / "pixel_widths.npz"
if widths_cache.exists():
    widths = np.load(widths_cache)["widths"].astype(np.float32)
    print(f"Loaded cached widths: {len(widths):,} pixels")
else:
    def _gt_gen():
        for p in val_pairs:
            if is_cached(p.sample_id, BASELINE_CACHE_DIR):
                _, gt = load_cached(p.sample_id, BASELINE_CACHE_DIR)
                yield gt
    widths = collect_pixel_widths(_gt_gen(), max_samples=5_000_000)
    np.savez_compressed(widths_cache, widths=widths.astype(np.float16))
    print(f"Computed widths: {len(widths):,} pixels")

pct_edges = percentile_edges(widths, n_buckets=3)
print(f"Fixed edges:      {FIXED_EDGES} px")
print(f"Percentile edges: ({pct_edges[0]:.1f}, {pct_edges[1]:.1f}) px")


---
## 6 · Width-stratified IoU — both models side-by-side

Showing **fixed-edge FP-in** as the headline (full IoU per bucket,
interpretable thresholds). Percentile variant computed too.


In [ ]:
import pandas as pd
from tqdm import tqdm
from road_segmentation.analysis.width_metrics import (
    stratify_sample, aggregate_bucket_stats, format_bucket_table,
    fixed_bucket, percentile_bucket,
)

THRESHOLD = 0.5
pct_bucket = lambda w: percentile_bucket(w, pct_edges)  # noqa: E731

def run_stratified(cache_dir, label):
    fixed_in, fixed_out, pct_in, pct_out = [], [], [], []
    for pair in tqdm(val_pairs, desc=f"[{label}] stratify"):
        if not is_cached(pair.sample_id, cache_dir):
            continue
        prob, gt = load_cached(pair.sample_id, cache_dir)
        f_in, f_out = stratify_sample(prob, gt, THRESHOLD, fixed_bucket, n_buckets=3)
        p_in, p_out = stratify_sample(prob, gt, THRESHOLD, pct_bucket, n_buckets=3)
        fixed_in.append(f_in); fixed_out.append(f_out)
        pct_in.append(p_in);   pct_out.append(p_out)
    return {
        "fixed_fp_in":  aggregate_bucket_stats(fixed_in),
        "fixed_fp_out": aggregate_bucket_stats(fixed_out),
        "pct_fp_in":    aggregate_bucket_stats(pct_in),
        "pct_fp_out":   aggregate_bucket_stats(pct_out),
    }

baseline_stats = run_stratified(BASELINE_CACHE_DIR, "baseline")
cldice_stats   = run_stratified(CLDICE_CACHE_DIR, "clDice")


In [ ]:
fixed_labels = [f"narrow (≤{FIXED_EDGES[0]:.0f})",
                f"medium ({FIXED_EDGES[0]:.0f}-{FIXED_EDGES[1]:.0f})",
                f"wide (>{FIXED_EDGES[1]:.0f})"]
pct_labels = [f"narrow (≤{pct_edges[0]:.1f})",
              f"medium ({pct_edges[0]:.1f}-{pct_edges[1]:.1f})",
              f"wide (>{pct_edges[1]:.1f})"]

def side_by_side(base, cld, labels, title):
    a = pd.DataFrame(format_bucket_table(base, labels)).set_index("bucket")
    b = pd.DataFrame(format_bucket_table(cld, labels)).set_index("bucket")
    delta = (b[["iou", "precision", "recall"]] - a[["iou", "precision", "recall"]]).round(4)
    out = pd.concat(
        {"baseline": a[["iou", "precision", "recall"]],
         "clDice":   b[["iou", "precision", "recall"]],
         "Δ":        delta},
        axis=1,
    )
    print(f"\n=== {title} ===")
    return out

display(side_by_side(baseline_stats["fixed_fp_in"],  cldice_stats["fixed_fp_in"],  fixed_labels, "Fixed thresholds, FP-in (full IoU)"))
display(side_by_side(baseline_stats["fixed_fp_out"], cldice_stats["fixed_fp_out"], fixed_labels, "Fixed thresholds, FP-out (recall only)"))
display(side_by_side(baseline_stats["pct_fp_in"],    cldice_stats["pct_fp_in"],    pct_labels,   "Percentile buckets, FP-in"))


---
## 7 · Delta bar chart — per-bucket IoU improvement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(3); w = 0.35

for ax, (title, base, cld, labels) in zip(
    axes,
    [("Fixed thresholds",    baseline_stats["fixed_fp_in"], cldice_stats["fixed_fp_in"], fixed_labels),
     ("Percentile buckets",  baseline_stats["pct_fp_in"],   cldice_stats["pct_fp_in"],   pct_labels)],
):
    base_iou = [base[i].iou() for i in range(3)]
    cld_iou  = [cld[i].iou()  for i in range(3)]
    b1 = ax.bar(x - w/2, base_iou, w, label="baseline", color="#2b8cbe", edgecolor="black")
    b2 = ax.bar(x + w/2, cld_iou,  w, label="clDice",   color="#41ae76", edgecolor="black")
    for bars, vals in [(b1, base_iou), (b2, cld_iou)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.015, f"{v:.3f}",
                    ha="center", fontsize=9)
    for i, (bv, cv) in enumerate(zip(base_iou, cld_iou)):
        delta = cv - bv
        color = "#238b45" if delta > 0 else "#cb181d"
        ax.text(i, max(bv, cv) + 0.06, f"Δ={delta:+.3f}",
                ha="center", fontweight="bold", color=color)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=0, fontsize=9)
    ax.set_ylabel("IoU"); ax.set_ylim(0, 1.05)
    ax.set_title(title); ax.legend(loc="lower right")

plt.suptitle("Per-bucket IoU: baseline vs clDice fine-tune", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Global IoU numbers
def global_iou(stats): 
    tp = sum(stats[i].tp for i in range(3))
    fp = sum(stats[i].fp for i in range(3))
    fn = sum(stats[i].fn for i in range(3))
    return tp / max(tp + fp + fn, 1)

print(f"\nGlobal IoU — baseline: {global_iou(baseline_stats['fixed_fp_in']):.4f}")
print(f"Global IoU — clDice:   {global_iou(cldice_stats['fixed_fp_in']):.4f}")


---
## 8 · Connectivity — baseline vs clDice

In [ ]:
from road_segmentation.analysis.connectivity import analyze_sample, aggregate

def run_connectivity(cache_dir, label):
    results = []
    for pair in tqdm(val_pairs, desc=f"[{label}] connectivity"):
        if not is_cached(pair.sample_id, cache_dir):
            continue
        prob, gt = load_cached(pair.sample_id, cache_dir)
        pred = (prob >= THRESHOLD).astype(np.uint8)
        results.append(analyze_sample(gt, pred, sample_id=pair.sample_id, min_fragment_pct=0.10))
    return results, aggregate(results)

baseline_conn, baseline_conn_summary = run_connectivity(BASELINE_CACHE_DIR, "baseline")
cldice_conn,   cldice_conn_summary   = run_connectivity(CLDICE_CACHE_DIR, "clDice")

conn_df = pd.DataFrame({
    "baseline": baseline_conn_summary,
    "clDice":   cldice_conn_summary,
})
conn_df["Δ"] = conn_df["clDice"] - conn_df["baseline"]
display(conn_df.round(4))


In [ ]:
# Paired per-sample deltas (same sample_id order both runs)
base_by_id = {r.sample_id: r for r in baseline_conn}
cld_by_id  = {r.sample_id: r for r in cldice_conn}
common_ids = [sid for sid in base_by_id if sid in cld_by_id and base_by_id[sid].n_gt_components > 0]

d_conn = np.array([cld_by_id[sid].connectivity_fraction - base_by_id[sid].connectivity_fraction
                   for sid in common_ids])

print(f"Per-sample connectivity delta over {len(common_ids)} samples:")
print(f"  mean   Δ = {d_conn.mean():+.4f}")
print(f"  median Δ = {np.median(d_conn):+.4f}")
print(f"  %improved = {(d_conn > 0).mean() * 100:.1f}%")
print(f"  %worse    = {(d_conn < 0).mean() * 100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(d_conn, bins=41, color="#41ae76", edgecolor="white", alpha=0.85)
ax.axvline(0, color="black", ls="-", lw=1.5)
ax.axvline(d_conn.mean(), color="red", ls="--", lw=2, label=f"mean = {d_conn.mean():+.3f}")
ax.set_xlabel("Δ connectivity fraction (clDice − baseline)")
ax.set_ylabel("Sample count")
ax.set_title("Per-sample connectivity change")
ax.legend()
plt.tight_layout(); plt.show()


---
## 9 · Side-by-side failure gallery

In [ ]:
import cv2
from PIL import Image as PILImage

# Pick samples where clDice helps most on connectivity
deltas_sorted = sorted(
    [(sid, cld_by_id[sid].connectivity_fraction - base_by_id[sid].connectivity_fraction,
      base_by_id[sid].n_gt_components)
     for sid in common_ids if base_by_id[sid].n_gt_components >= 3],
    key=lambda t: -t[1],
)[:5]

pair_by_id = {p.sample_id: p for p in val_pairs}
N = len(deltas_sorted)

fig, axes = plt.subplots(N, 3, figsize=(15, 4.5 * N))
if N == 1: axes = axes[np.newaxis, :]

def overlay(image, gt_b, pred_b, alpha=0.55):
    out = image.copy().astype(np.float32)
    out[gt_b & pred_b]   = (1 - alpha) * out[gt_b & pred_b]   + alpha * np.array([0, 255, 0])
    out[pred_b & ~gt_b]  = (1 - alpha) * out[pred_b & ~gt_b]  + alpha * np.array([255, 0, 0])
    out[~pred_b & gt_b]  = (1 - alpha) * out[~pred_b & gt_b]  + alpha * np.array([0, 0, 255])
    return np.clip(out, 0, 255).astype(np.uint8)

for row_idx, (sid, delta, _) in enumerate(deltas_sorted):
    pair = pair_by_id[sid]
    image = np.array(PILImage.open(pair.image_path).convert("RGB"))
    b_prob, gt = load_cached(sid, BASELINE_CACHE_DIR)
    c_prob, _  = load_cached(sid, CLDICE_CACHE_DIR)
    gt_b = gt.astype(bool)
    b_pred = (b_prob >= THRESHOLD).astype(bool)
    c_pred = (c_prob >= THRESHOLD).astype(bool)

    axes[row_idx, 0].imshow(image); axes[row_idx, 0].axis("off")
    axes[row_idx, 0].set_title(f"Input — {sid[-12:]} (Δconn={delta:+.2f})", fontsize=10)

    axes[row_idx, 1].imshow(overlay(image, gt_b, b_pred)); axes[row_idx, 1].axis("off")
    axes[row_idx, 1].set_title(
        f"Baseline — conn={base_by_id[sid].connectivity_fraction:.2f}, "
        f"pred_cc={base_by_id[sid].n_pred_components}", fontsize=10)

    axes[row_idx, 2].imshow(overlay(image, gt_b, c_pred)); axes[row_idx, 2].axis("off")
    axes[row_idx, 2].set_title(
        f"clDice — conn={cld_by_id[sid].connectivity_fraction:.2f}, "
        f"pred_cc={cld_by_id[sid].n_pred_components}", fontsize=10)

plt.suptitle("Samples where clDice helped most — TP green / FP red / FN blue",
             fontsize=13, fontweight="bold", y=1.002)
plt.tight_layout(); plt.show()


---
## 10 · Takeaways

Fill these in after running the notebook. Template:

1. **Narrow-bucket IoU** baseline X.XXX → clDice Y.YYY (Δ = +Z.ZZZ).
   Fixed-edge FP-in variant, threshold 0.5.

2. **Connectivity fraction** baseline X.XXX → clDice Y.YYY (Δ = +Z.ZZZ).
   P% of samples improved, Q% got worse.

3. **Aggregate IoU** baseline X.XXX → clDice Y.YYY.
   If this dropped slightly while narrow-bucket IoU rose, that's
   the intended trade-off: we reweighted the loss toward thin roads
   and connectivity, at a small cost to pixel-mass metrics. The
   product (routing) cares about the new number, not the old one.

4. **Loss design:** clDice_iters = 5 chosen to match the ≤ 10 px
   narrow-bucket cutoff from the width analysis. Each erosion
   iteration peels ~2 px, so the skeleton emerges for every road
   we under-performed on.

5. **Branch discipline:** all of this lives on `metric-analysis`;
   the submitted model on `main` is untouched.
